# Week 6 — Dirty Data Pipeline

> **There is nothing to complete in this notebook.** No `TODO`, no grade, no `RESULTS` block. Your job is to **run it top to bottom, read it, and keep one file it produces** (`feature_table.parquet`). Week 7 starts from that file.

## The story

Until now the data was polite. MovieLens arrives as clean tables, every rating is a number, every timestamp is in seconds, and nobody rated the same movie twice. A real app does not work like that. Its **event log** is written by phones, browsers and retrying servers, and it contains:

- the same event logged twice (or logged again a few seconds later, because a request was retried);
- half-written rows with a missing user or rating;
- timestamps in milliseconds, in text form, in 1970 or in 2065;
- numbers stored as text (`"4,5"`, `" 17 "`, `"296.0"`);
- rows about items that do not exist;
- and one column that quietly contains **the future**.

Today you watch a pipeline turn such a log into a **feature table**: the clean, reproducible table that Week 7 uses to ask *where does our model fail?*

To make the story checkable, we do something you can only do in a classroom: we take the clean MovieLens ratings, **break them on purpose with a provided script** (fixed seed, so everyone gets the same mess), and then clean them with `pandas`. At the end we prove that we recovered the original data, row for row.

## What you will see

| Step | Question it answers | pandas tools |
|---|---|---|
| 0 | What does a dirty log look like? | `read_csv`, `dtypes`, `isna`, `sample` |
| 1 | Are the types right? | `to_numeric`, `to_datetime`, `str` methods |
| 2 | Which rows are unusable? | `dropna` |
| 3 | Which timestamps are impossible? | `between` |
| 4 | Which rows point to items that do not exist? | `merge(indicator=True, validate=...)` |
| 5 | Which rows are the same event twice, and why does the order of steps matter? | `drop_duplicates`, `sort_values`, `groupby` |
| 6 | Which column knows the future? | leakage |
| 7 | Can we prove the result is right? | `assert_frame_equal` |
| 8 | How do events become features without peeking? | `cumcount`, `merge`, `cut`, `map` |
| 9 | Is the output the same for everybody? | a hash of the final table, `to_parquet` |

**Runtime:** a few minutes on CPU. No GPU, no account, nothing to install by hand.

**No InstaKill work this week.** Nothing in your project repository changes. The habits (count before/after, check your assumptions, never use the future) apply to every data pipeline you will write.

In [1]:
%pip -q install numpy pandas pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import hashlib
import time
import zipfile
from collections import defaultdict, deque, namedtuple
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
DAY = 86_400                    # seconds in a day
K = 10                          # a feed shows 10 items
CANDIDATES = 99                 # the exam: 1 liked item hidden among 99 random items

# --- the Week 5 constants, unchanged: the feature table must be built with exactly the same rules ---
HALF_LIFE_DAYS = 3 * 365
POPULARITY_DAYS = 365
ACTIVITY_CAP = 500
AGE_CAP_DAYS = 25 * 365
MIN_HISTORY = 3
NEGATIVES_PER_LIKE = 4
EVAL_MOMENTS = 2_000

FEATURES = ["freshness", "affinity", "popularity", "activity", "lifetime", "age"]
DATA_DIR = Path("ml-latest-small")

# --- two facts about the (pretend) log, as a data engineer would know them ---
MIN_VALID_TS = int(pd.Timestamp("1995-01-01", tz="UTC").timestamp())   # the platform did not exist before this
EXPORT_TS = int(pd.Timestamp("2018-09-25", tz="UTC").timestamp())      # the log was exported on this day: nothing can happen later

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.6 | numpy 2.5.4


## Data

Same dataset as Weeks 4 and 5: MovieLens `latest-small`, about 100 000 ratings from 610 users. It comes from GroupLens and is **not** stored in the course repository. Read its terms of use in `README.txt` inside the zip.

**Download it in your browser:** <https://files.grouplens.org/datasets/movielens/ml-latest-small.zip> (you may still have it from Week 4 or 5). Then run the next cell: in Colab it asks you to upload the zip; locally, put the zip next to this notebook. (An automatic download from Python fails on GroupLens's certificate chain.)

In [3]:
if not (DATA_DIR / "ratings.csv").exists():
    archives = sorted(Path(".").glob("ml-latest-small*.zip"))
    if not archives:
        try:
            from google.colab import files
            print("Select the ml-latest-small.zip file you downloaded.")
            files.upload()
        except ImportError:
            raise FileNotFoundError("Download ml-latest-small.zip (link above) and place it next to this notebook.")
        archives = sorted(Path(".").glob("ml-latest-small*.zip"))
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(".")
for name in ("ratings.csv", "movies.csv"):
    assert (DATA_DIR / name).exists(), f"{DATA_DIR / name} not found: upload the original ml-latest-small.zip"

# TRUTH is the clean table. A real pipeline never has it; we have it only because we are about to break it ourselves.
TRUTH = pd.read_csv(DATA_DIR / "ratings.csv").rename(columns={"userId": "user_id", "movieId": "item_id", "timestamp": "ts"})
MOVIES_DF = pd.read_csv(DATA_DIR / "movies.csv").rename(columns={"movieId": "item_id"})
print(f"{len(TRUTH):,} ratings, {TRUTH['user_id'].nunique()} users, {len(MOVIES_DF):,} movies")
TRUTH.head(3)

100,836 ratings, 610 users, 9,742 movies


,user_id,item_id,rating,ts
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224


## Step 0 — Make the mess (provided script) and look at it

The function below writes `raw_events.csv`, a dirty version of the ratings. **You do not need to read it**, but it is short and it is the list of everything that goes wrong in this notebook. Each kind of damage is either *fixable* (the true value can be recovered) or an *extra row* that should be thrown away (a copy of a real event that went wrong). That is why a perfect cleaning is possible, and checkable, at the end.

| Damage | How much | What the pipeline must do |
|---|---:|---|
| timestamp in milliseconds | 2 % of rows | divide by 1000 |
| timestamp as date text (`2015-03-01 12:00:00`) | 2 % | parse as UTC |
| rating as text (`"4,5"`, `" 4.0 "`) | 3 % | fix the decimal comma, strip spaces |
| ids as text (`" 17 "`, `"296.0"`) | 2 % | strip, convert to integer |
| the same row logged twice, byte for byte | 2 % extra rows | drop the copy |
| a retry: same event, 1–60 seconds later | 1.5 % extra rows | keep the first |
| a half-written copy (rating or user missing) | 1.5 % extra rows | drop the copy |
| a copy with timestamp `0` or in the year 2065 | 1 % extra rows | drop the copy |
| a copy about an item that does not exist | 0.5 % extra rows | drop the copy |
| a column that knows the future (`item_total_likes`) | every row | **never use it** |

In [4]:
def make_dirty_log(truth, seed=SEED):
    """Provided. Break the clean ratings in the ways the table above lists. Deterministic for a given seed."""
    rng = np.random.default_rng(seed)
    n = len(truth)
    likes_total = truth[truth["rating"] >= 4.0].groupby("item_id").size()     # computed on the WHOLE history: it contains the future
    clean = pd.DataFrame({
        "user_id": truth["user_id"].astype(str),
        "item_id": truth["item_id"].astype(str),
        "rating": truth["rating"].map(lambda r: f"{r:.1f}"),
        "ts": truth["ts"].astype(str),
        "item_total_likes": truth["item_id"].map(likes_total).fillna(0).astype(int),
    })
    log = clean.copy()

    order, cursor = rng.permutation(n), 0
    def block(fraction):                       # a fresh set of rows that no other in-place damage touches
        nonlocal cursor
        idx = order[cursor:cursor + int(n * fraction)]
        cursor += len(idx)
        return idx

    idx = block(0.02)                                                           # timestamps in milliseconds
    log.loc[idx, "ts"] = (truth.loc[idx, "ts"] * 1000).astype(str)
    idx = block(0.02)                                                           # timestamps as date text (UTC)
    log.loc[idx, "ts"] = pd.to_datetime(truth.loc[idx, "ts"], unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M:%S")
    idx = block(0.015)                                                          # decimal comma
    log.loc[idx, "rating"] = log.loc[idx, "rating"].str.replace(".", ",", regex=False)
    idx = block(0.015)                                                          # stray spaces
    log.loc[idx, "rating"] = " " + log.loc[idx, "rating"] + " "
    idx = block(0.01)                                                           # ids with spaces
    log.loc[idx, "user_id"] = " " + log.loc[idx, "user_id"] + " "
    idx = block(0.01)                                                           # ids that went through a float
    log.loc[idx, "item_id"] = log.loc[idx, "item_id"] + ".0"

    def copies(source, fraction):
        return source.iloc[rng.choice(len(source), int(n * fraction), replace=False)].copy()

    duplicates = copies(log, 0.02)                                              # byte-for-byte repeats
    retries = copies(clean, 0.015)                                              # same event, logged again a little later
    retries["ts"] = (retries["ts"].astype(int) + rng.integers(1, 61, len(retries))).astype(str)
    half_rating = copies(clean, 0.01); half_rating["rating"] = ""               # half-written rows
    half_user = copies(clean, 0.005); half_user["user_id"] = ""
    ts_zero = copies(clean, 0.005); ts_zero["ts"] = "0"                         # 1970
    ts_future = copies(clean, 0.005); ts_future["ts"] = (ts_future["ts"].astype(int) + 1_500_000_000).astype(str)   # ~2065
    ghosts = copies(clean, 0.005); ghosts["item_id"] = (900_000 + rng.integers(0, 5_000, len(ghosts))).astype(str)  # unknown items

    dirty = pd.concat([log, duplicates, retries, half_rating, half_user, ts_zero, ts_future, ghosts])
    return dirty.iloc[rng.permutation(len(dirty))].reset_index(drop=True)      # the log is in no particular order


make_dirty_log(TRUTH).to_csv("raw_events.csv", index=False)
print(f"wrote raw_events.csv ({Path('raw_events.csv').stat().st_size / 1e6:.1f} MB)")

wrote raw_events.csv (3.0 MB)


### First look

From here on we pretend we only have `raw_events.csv`, which is the normal situation. The first habit of a data engineer is to **look before touching**: shape, types, missing values, a few rows.

In [5]:
raw = pd.read_csv("raw_events.csv")
print(f"{len(raw):,} rows (the clean table had {len(TRUTH):,})\n")
print(raw.dtypes, "\n")
print("missing values per column:")
print(raw.isna().sum())
raw.sample(8, random_state=1)

107,388 rows (the clean table had 100,836)

user_id             float64
item_id             float64
rating                  str
ts                      str
item_total_likes      int64
dtype: object 

missing values per column:
user_id              504
item_id                0
rating              1008
ts                     0
item_total_likes       0
dtype: int64


,user_id,item_id,rating,ts,item_total_likes
26330,393.0,49272.0,5.0,1430506696,52
103972,125.0,71057.0,4.0,1474482104,8
84264,266.0,1240.0,4.0,945670292,94
21616,182.0,543.0,3.5,1054779684,15
50689,226.0,46578.0,5.0,1159219237,50
70881,474.0,36276.0,3.5,1165539993,0
51204,470.0,2.0,3.0,2349224778,50
25980,68.0,34162.0,5.0,1158532330,25


Read the `dtypes` output carefully. `user_id`, `item_id`, `rating` and `ts` should be numbers, but at least some of them are `object` or `str`, which means pandas found text it could not read as a number. That is the first alarm: **a numeric column that is not numeric is dirty.**

A few quick counts show what kind of text hides in the timestamp column:

In [6]:
ts_text = raw["ts"].astype(str).str.strip()
looks_like_date = ts_text.str.contains("-", regex=False)
numeric_ts = pd.to_numeric(ts_text.where(~looks_like_date), errors="coerce")
print(f"timestamps written as a date:        {looks_like_date.sum():>7,}")
print(f"timestamps that look like seconds:   {(numeric_ts < 1e11).sum():>7,}")
print(f"timestamps that look like millis:    {(numeric_ts >= 1e11).sum():>7,}")
print(f"timestamps equal to 0 (1970):        {(numeric_ts == 0).sum():>7,}")
print(f"timestamps after the export day:     {((numeric_ts > EXPORT_TS) & (numeric_ts < 1e11)).sum():>7,}")

timestamps written as a date:          2,046
timestamps that look like seconds:   103,291
timestamps that look like millis:      2,051
timestamps equal to 0 (1970):            504
timestamps after the export day:         504


## The audit trail

Every cleaning step below prints **how many rows it removed**. A pipeline that changes the data without telling you how much is a pipeline you cannot trust. The `record` helper collects all the counts, and the last cell of the notebook prints them as one table.

In [7]:
AUDIT = []

def record(step, before, after):
    AUDIT.append({"step": step, "rows before": before, "rows after": after, "removed": before - after})
    print(f"{step:<42} {before:>9,} -> {after:>9,}   ({before - after:,} removed)")

## Step 1 — Fix the types

Rule of thumb: **convert first, filter later.** You cannot ask "is this timestamp in the future?" about a piece of text.

- `str.strip()` removes spaces, `str.replace(",", ".")` fixes the decimal comma;
- `pd.to_numeric(..., errors="coerce")` turns anything unreadable into `NaN` instead of crashing, so that the next step can find it;
- a timestamp can arrive in three forms; we normalise all of them to **seconds since 1970 (UTC)**.

In [8]:
def to_number(column):
    """Text -> float. Spaces are stripped, a decimal comma becomes a point, unreadable values become NaN."""
    text = column.astype(str).str.strip().str.replace(",", ".", regex=False)
    return pd.to_numeric(text, errors="coerce").astype("float64")


def to_epoch_seconds(column):
    """Timestamps in seconds, in milliseconds or as date text -> seconds since 1970 (UTC), NaN if unreadable."""
    text = column.astype(str).str.strip()
    is_date = text.str.contains("-", regex=False)
    seconds = to_number(text.where(~is_date))                       # the numeric ones
    seconds = seconds.where(seconds < 1e11, seconds / 1000)         # 1e11 seconds would be the year 5138: those are milliseconds
    dates = pd.to_datetime(text.where(is_date), format="%Y-%m-%d %H:%M:%S", utc=True, errors="coerce")
    from_dates = (dates - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds()
    return seconds.fillna(from_dates)


df = raw.copy()
for name in ("user_id", "item_id", "rating"):
    df[name] = to_number(df[name])
df["ts"] = to_epoch_seconds(df["ts"])

print(df.dtypes, "\n")
print("missing values per column (these are the rows that were half-written):")
print(df.isna().sum())
record("1. fix types (nothing removed)", len(raw), len(df))

user_id             float64
item_id             float64
rating              float64
ts                  float64
item_total_likes      int64
dtype: object 

missing values per column (these are the rows that were half-written):
user_id              504
item_id                0
rating              1008
ts                     0
item_total_likes       0
dtype: int64
1. fix types (nothing removed)               107,388 ->   107,388   (0 removed)


## Step 2 — Drop the rows that cannot be used

A row without a user, an item, a rating or a time is not an event we can learn from. We do not try to guess the missing value: here the half-written rows are copies of real events that are present elsewhere in the log, so dropping them loses nothing.

*Question to ask in a real project: "does dropping these rows bias my data?" Here it does not. If, say, only mobile users had missing ratings, it would.*

In [9]:
before = len(df)
df = df.dropna(subset=["user_id", "item_id", "rating", "ts"])
for name in ("user_id", "item_id", "ts"):
    df[name] = df[name].astype("int64")                              # now it is safe to use integers
record("2. drop incomplete rows", before, len(df))

2. drop incomplete rows                      107,388 ->   105,876   (1,512 removed)


## Step 3 — Drop impossible timestamps

Two facts we know about this log: the platform did not exist before 1995, and the log was exported on 2018-09-25, so nothing can have happened later. A timestamp outside that range is a *bug in the data*, not an opinion. We keep a snapshot of the table before this step, because the next-but-one cell shows **why the order of steps matters**.

In [10]:
before_impossible_filter = df.copy()

before = len(df)
df = df[df["ts"].between(MIN_VALID_TS, EXPORT_TS)]
record("3. drop impossible timestamps", before, len(df))

3. drop impossible timestamps                105,876 ->   104,868   (1,008 removed)


## Step 4 — Drop events about items that do not exist

Every event must point to a movie that is in `movies.csv`. A `merge` with `indicator=True` adds a column that says, row by row, whether a match was found. `validate="m:1"` makes pandas **check our assumption** that each item appears once in the movies table, instead of silently multiplying rows (a classic join bug).

In [11]:
before = len(df)
checked = df.merge(MOVIES_DF[["item_id"]], on="item_id", how="left", indicator=True, validate="m:1")
print(checked["_merge"].value_counts().to_string(), "\n")
df = checked[checked["_merge"] == "both"].drop(columns="_merge")
record("4. drop items not in movies.csv", before, len(df))

_merge
both          104364
left_only        504
right_only         0 

4. drop items not in movies.csv              104,868 ->   104,364   (504 removed)


## Step 5 — Duplicates: two different problems

**Exact repeats** (every column equal) are easy: `drop_duplicates()`.

**Retries** are harder: the same user rated the same movie, but the second copy was logged a few seconds later. The rows are not identical, yet they are one event. The rule is *one row per (user, item)*, and we keep the **earliest**, because the first write is the real one. Before applying a rule, we check the assumption behind it: did any retry change the rating? If it had, "keep the first" could hide a conflict.

In [12]:
before = len(df)
df = df.drop_duplicates()
record("5a. drop exact repeats", before, len(df))

ratings_per_pair = df.groupby(["user_id", "item_id"])["rating"].nunique()
print(f"\n(user, item) pairs logged more than once: {(df.groupby(['user_id', 'item_id']).size() > 1).sum():,}")
assert ratings_per_pair.max() == 1, "a retry changed the rating: 'keep the first' would hide a conflict, look at the data"
print("assumption holds: no retry changed the rating\n")

before = len(df)
df = df.sort_values(["ts", "user_id", "item_id"]).drop_duplicates(subset=["user_id", "item_id"], keep="first")
record("5b. keep the first event per (user, item)", before, len(df))

5a. drop exact repeats                       104,364 ->   102,348   (2,016 removed)

(user, item) pairs logged more than once: 1,512
assumption holds: no retry changed the rating

5b. keep the first event per (user, item)    102,348 ->   100,836   (1,512 removed)


### Why the order of the steps matters

In Step 3 we removed timestamps in 1970 *before* looking for retries. What if we had not? Copies with `ts = 0` look like the "earliest" event, so "keep the first" would pick **the broken copy** and throw away the real one. The experiment below runs Steps 4–5 on the snapshot we saved before Step 3, without Step 3, and counts the damage.

In [13]:
wrong_order = before_impossible_filter.drop_duplicates()
wrong_order = wrong_order[wrong_order["item_id"].isin(MOVIES_DF["item_id"])]
wrong_order = wrong_order.sort_values(["ts", "user_id", "item_id"]).drop_duplicates(subset=["user_id", "item_id"], keep="first")

truth_ts = TRUTH.set_index(["user_id", "item_id"])["ts"]
wrong_ts = wrong_order.set_index(["user_id", "item_id"])["ts"]
broken = (wrong_ts != truth_ts.reindex(wrong_ts.index)).sum()
right_ts = df.set_index(["user_id", "item_id"])["ts"]
fine = (right_ts != truth_ts.reindex(right_ts.index)).sum()
print(f"Dedup before the timestamp filter: {broken:,} events end up with a wrong timestamp (1970 or 2065).")
print(f"Dedup after the timestamp filter:  {fine:,}")

Dedup before the timestamp filter: 504 events end up with a wrong timestamp (1970 or 2065).
Dedup after the timestamp filter:  0


**Lesson:** each step assumes the previous ones did their job. Write the steps in the order in which their assumptions become true: types, then validity, then references, then duplicates.

## Step 6 — The column that knows the future

Everything above was *dirty*. This column is different: it is perfectly clean, and it is poison. `item_total_likes` was exported at the end of the dataset, so it counts **every like an item will ever get, including likes that have not happened yet** at the time of the event. A model that uses it looks brilliant in the lab and is useless in the real app, which can only know the past.

Let us see how tempting it is. Take the very first like of the most-liked movie in the log: compare what the app could have known at that moment with what this column says:

In [14]:
liked_events = df[df["rating"] >= 4.0]
prior = liked_events.groupby("item_id").cumcount()                       # likes of the same item before this one (df is in time order)
example = liked_events.loc[(liked_events["item_total_likes"] - prior).idxmax()]     # the biggest gap between "past" and "what the column says"
print(f"Movie {int(example['item_id'])}, at the moment of this like ({pd.to_datetime(int(example['ts']), unit='s').date()}):")
print(f"  likes the app could have counted:      {int(prior.loc[example.name])}")
print(f"  what item_total_likes says:            {int(example['item_total_likes'])}   <- includes the future")

Movie 318, at the moment of this like (1996-04-17):
  likes the app could have counted:      0
  what item_total_likes says:            274   <- includes the future


In [15]:
LEAKY_TOTAL_LIKES = df.groupby("item_id")["item_total_likes"].max()     # a copy kept ONLY to demonstrate the trap below
df = df.drop(columns="item_total_likes")
print("columns the pipeline is allowed to use from now on:", list(df.columns))

columns the pipeline is allowed to use from now on: ['user_id', 'item_id', 'rating', 'ts']


## Step 7 — Did we get it right? Compare with the truth

In real life nobody hands you the answer. Here we can, so this is the strongest possible test: after all the cleaning, the table must be **identical** to the original MovieLens ratings, every row and every value. `assert_frame_equal` raises an error with the first difference if it is not.

In [16]:
CLEAN = (df[["user_id", "item_id", "rating", "ts"]]
         .sort_values(["ts", "user_id", "item_id"]).reset_index(drop=True))
EXPECTED = TRUTH[["user_id", "item_id", "rating", "ts"]].sort_values(["ts", "user_id", "item_id"]).reset_index(drop=True)

pd.testing.assert_frame_equal(CLEAN, EXPECTED)
print(f"OK: the cleaned log has {len(CLEAN):,} events and is identical to the original ratings.")
print("A real pipeline has no such answer key: that is why we count rows at every step and test every assumption we can.")

OK: the cleaned log has 100,836 events and is identical to the original ratings.
A real pipeline has no such answer key: that is why we count rows at every step and test every assumption we can.


## Step 8 — From clean events to features, without peeking at the future

The feature table has to be built with **exactly the signals the Week 5 model was trained on** (freshness, affinity, popularity, activity, lifetime, age). If we rewrote the formulas, tiny differences would silently change the numbers the model sees, which is called **train/serve skew**. So we do not rewrite them: the next two cells are the Week 5 code, unchanged, and the only new thing is that they now read the **clean** events produced above.

The idea is the same as in Week 5: every like is a *moment* in which a user picked an item. For each moment of the `val` and `test` periods we hide the liked item among 99 random items that existed at that moment (the **exam**), and compute the six signals for all 100 candidates using **only what happened before that moment**.

In [17]:
# --- Week 5 code, unchanged, except that its input now comes from the clean pandas table ---
EVENTS = list(zip(CLEAN["user_id"], CLEAN["item_id"], CLEAN["ts"], CLEAN["rating"]))
MOVIES = {int(r.item_id): (r.title, tuple(g for g in r.genres.split("|") if g != "(no genres listed)"))
          for r in MOVIES_DF.itertuples(index=False)}

first_seen = {}
for _, movie_id, ts, _ in EVENTS:
    first_seen[movie_id] = min(ts, first_seen.get(movie_id, ts))
movie_ids = sorted(first_seen, key=lambda m: (first_seen[m], m))      # oldest first
ROW = {m: row for row, m in enumerate(movie_ids)}                     # movie id -> row number

GENRES = sorted({g for m in movie_ids for g in MOVIES.get(m, ("?", ()))[1]})
genre_matrix = np.zeros((len(movie_ids), len(GENRES)))
for row, m in enumerate(movie_ids):
    for g in MOVIES.get(m, ("?", ()))[1]:
        genre_matrix[row, GENRES.index(g)] = 1.0

Catalogue = namedtuple("Catalogue", "titles first_seen genres n_genres")
CATALOGUE = Catalogue(
    titles=[MOVIES.get(m, ("?", ()))[0] for m in movie_ids],
    first_seen=np.array([first_seen[m] for m in movie_ids]),
    genres=genre_matrix,
    n_genres=np.maximum(genre_matrix.sum(axis=1), 1.0),
)

Like = namedtuple("Like", "user_id row ts")
LIKES = sorted((Like(u, ROW[m], ts) for u, m, ts, rating in EVENTS if rating >= 4.0),
               key=lambda like: (like.ts, like.user_id, like.row))
N_USERS = max(like.user_id for like in LIKES) + 1

timestamps = np.array([like.ts for like in LIKES])
T_VAL, T_TEST = int(np.quantile(timestamps, 0.70)), int(np.quantile(timestamps, 0.85))
print(f"{len(LIKES):,} likes, {len(movie_ids):,} items, {len({l.user_id for l in LIKES})} users, {len(GENRES)} genres")

48,580 likes, 9,724 items, 609 users, 19 genres


In [18]:
class LiveSignals:
    """What we know about the world right now. Feed it likes in time order with add_like()."""

    def __init__(self, catalogue, n_users):
        self.catalogue = catalogue
        self.genre_counts = np.zeros((n_users, len(GENRES)))   # row u: how many liked items of each genre user u has
        self.n_likes = np.zeros(n_users)                       # likes per user
        self.liked = defaultdict(set)                          # user -> set of item rows already liked
        self.recent = np.zeros(len(catalogue.titles))          # per item: likes in the last POPULARITY_DAYS days
        self.lifetime = np.zeros(len(catalogue.titles))        # per item: likes ever
        self._window = deque()                                 # (ts, row) of the likes counted in `recent`

    def add_like(self, like):
        self.genre_counts[like.user_id] += self.catalogue.genres[like.row]   # one vector addition, not a loop over genres
        self.n_likes[like.user_id] += 1
        self.liked[like.user_id].add(like.row)
        self.recent[like.row] += 1
        self.lifetime[like.row] += 1
        self._window.append((like.ts, like.row))

    def advance(self, now):
        """Forget the likes that are older than POPULARITY_DAYS days."""
        while self._window and self._window[0][0] < now - POPULARITY_DAYS * DAY:
            _, row = self._window.popleft()
            self.recent[row] -= 1

    def features(self, user_id, rows, now):
        """The 6 signals of user `user_id` for the items `rows` (an int array), as a (len(rows), 6) array."""
        cat = self.catalogue
        age_days = (now - cat.first_seen[rows]) / DAY
        freshness = 0.5 ** (age_days / HALF_LIFE_DAYS)

        counts = self.genre_counts[user_id]
        total = counts.sum()
        if total > 0:
            affinity = (cat.genres[rows] @ counts) / (total * cat.n_genres[rows])   # a matrix-vector product
        else:
            affinity = np.zeros(len(rows))                                           # nothing known about this user

        popularity = self.recent[rows] / max(self.recent.max(), 1)
        activity = np.full(len(rows), min(1.0, np.log1p(self.n_likes[user_id]) / np.log1p(ACTIVITY_CAP)))
        lifetime = np.log1p(self.lifetime[rows]) / np.log1p(max(self.lifetime.max(), 1))
        age = np.minimum(1.0, age_days / AGE_CAP_DAYS)
        return np.column_stack([freshness, affinity, popularity, activity, lifetime, age])

In [19]:
class Dataset:
    pass


def build_examples(likes, catalogue, n_users, seed=SEED):
    rng = np.random.default_rng(seed)
    ever_liked = defaultdict(set)
    for like in likes:
        ever_liked[like.user_id].add(like.row)

    def pick(low, high):                       # a fixed random subset of the moments in [low, high)
        idx = [k for k, like in enumerate(likes) if low <= like.ts < high]
        return set(rng.choice(idx, min(EVAL_MOMENTS, len(idx)), replace=False).tolist())

    exam = {"val": pick(T_VAL, T_TEST), "test": pick(T_TEST, 1 << 62)}

    signals = LiveSignals(catalogue, n_users)
    data = Dataset()
    train_x, train_y = [], []
    data.train_meta = []
    for name in exam:
        setattr(data, f"{name}_x", []); setattr(data, f"{name}_rows", []); setattr(data, f"{name}_meta", [])

    for k, like in enumerate(likes):
        signals.advance(like.ts)
        part = "train" if like.ts < T_VAL else ("val" if like.ts < T_TEST else "test")
        n_exist = int(np.searchsorted(catalogue.first_seen, like.ts, side="left"))   # items that existed before this moment
        wanted = NEGATIVES_PER_LIKE if part == "train" else (CANDIDATES if k in exam[part] else 0)
        if wanted and signals.n_likes[like.user_id] >= MIN_HISTORY and catalogue.first_seen[like.row] < like.ts and n_exist >= 200:
            negatives = []
            while len(negatives) < wanted:
                row = int(rng.integers(0, n_exist))
                if row not in ever_liked[like.user_id] and row not in negatives:
                    negatives.append(row)
            rows = np.array([like.row] + negatives)                  # the liked item is always entry 0
            x = signals.features(like.user_id, rows, like.ts)
            if part == "train":
                train_x.append(x); train_y.append(np.r_[1.0, np.zeros(wanted)]); data.train_meta.append(like)
            else:
                getattr(data, f"{part}_x").append(x); getattr(data, f"{part}_rows").append(rows); getattr(data, f"{part}_meta").append(like)
        signals.add_like(like)                                       # only now does the like become history

    data.train_x, data.train_y = np.vstack(train_x), np.concatenate(train_y)
    for name in exam:
        setattr(data, f"{name}_x", np.stack(getattr(data, f"{name}_x")))        # shape (moments, 100, 6)
        setattr(data, f"{name}_rows", np.stack(getattr(data, f"{name}_rows")))  # shape (moments, 100)
    return data


started = time.perf_counter()
DATA = build_examples(LIKES, CATALOGUE, N_USERS)
print(f"built in {time.perf_counter() - started:.1f} s")
print(f"train: {DATA.train_x.shape[0]:,} rows ({DATA.train_y.mean():.0%} positive), shape {DATA.train_x.shape}")
print(f"val:   {DATA.val_x.shape[0]:,} exam questions, shape {DATA.val_x.shape}  (moments, 1 liked + 99 random, signals)")
print(f"test:  {DATA.test_x.shape[0]:,} exam questions, shape {DATA.test_x.shape}")

built in 2.0 s
train: 147,825 rows (20% positive), shape (147825, 6)
val:   1,808 exam questions, shape (1808, 100, 6)  (moments, 1 liked + 99 random, signals)
test:  1,753 exam questions, shape (1753, 100, 6)


`DATA` holds numpy arrays, which is not what we want to hand to a colleague. A **feature table** is one row per candidate (a user, a moment, an item), with the six signals as columns and a `label` (1 for the liked item, 0 for the 99 random ones). This is the first place where pandas shines again: arrays go in, a table comes out.

In [20]:
def exam_to_frame(split, x, rows, meta, first_moment):
    """One exam split (arrays) -> one row per candidate. `x` is (moments, 100, 6); candidate 0 is the liked item."""
    moments, candidates, _ = x.shape
    frame = pd.DataFrame({
        "moment_id": np.repeat(np.arange(first_moment, first_moment + moments), candidates),
        "split": split,
        "user_id": np.repeat([like.user_id for like in meta], candidates),
        "ts": np.repeat([like.ts for like in meta], candidates),
        "like_row": np.repeat([like.row for like in meta], candidates),          # the item that was really liked in this moment
        "item_row": rows.reshape(-1),
        "label": np.tile(np.r_[1, np.zeros(candidates - 1, dtype=int)], moments),
    })
    for j, name in enumerate(FEATURES):
        frame[name] = x[:, :, j].reshape(-1)
    return frame


val = exam_to_frame("val", DATA.val_x, DATA.val_rows, DATA.val_meta, 0)
test = exam_to_frame("test", DATA.test_x, DATA.test_rows, DATA.test_meta, len(DATA.val_meta))
TABLE = pd.concat([val, test], ignore_index=True)
TABLE["item_id"] = np.array(movie_ids)[TABLE["item_row"]]                       # back from row numbers to movie ids
print(f"{len(TABLE):,} rows = {TABLE['moment_id'].nunique():,} exam questions x {CANDIDATES + 1} candidates")
TABLE.head(3)

356,100 rows = 3,561 exam questions x 100 candidates


,moment_id,split,user_id,ts,like_row,item_row,label,freshness,affinity,popularity,activity,lifetime,age,item_id
0,0,val,376,1364932581,198,198,1,0.020159,0.15,0.0,0.222999,0.518748,0.675891,376
1,0,val,376,1364932581,198,6046,0,0.387413,0.10,0.0,0.222999,0.000000,0.164167,65037
2,0,val,376,1364932581,198,598,0,0.023264,0.10,0.0,0.222999,0.459336,0.651094,1254


### Extra columns for Week 7 — computed with pandas, and cross-checked against the Week 5 code

Week 7 asks *where does the model fail?* To answer, it needs columns to **slice** the results by: new users vs heavy users, brand-new items vs old ones, by genre. We compute them with pandas operations, **independently** of the Week 5 code, and then use them to check that code. Two ways of computing the same number that agree is the cheapest test there is.

- `user_prior_likes`: how many likes the user had before this moment. `groupby(...).cumcount()` numbers the rows of each group 0, 1, 2, ... in time order.
- `item_prior_likes`: how many likes the candidate item had before this moment. For 400 000 rows at once we encode each like as `item * BIG + position` in time; counting earlier likes of an item is then two binary searches (`np.searchsorted`) on one sorted array.
- `item_age_days`: moment time minus the first time anybody rated the item (`groupby().min()` + `map`).
- `main_genre`, `n_genres`: from the pipe-separated genre string (`str.split`).
- `user_group`, `item_age_group`: `pd.cut` turns a number into named buckets.

In [21]:
likes_df = pd.DataFrame(LIKES)                                              # columns: user_id, row, ts (already in time order)
likes_df["pos"] = np.arange(len(likes_df))                                  # the position of the like in time
likes_df["user_prior_likes"] = likes_df.groupby("user_id").cumcount()       # 0 for a user's first like, 1 for the second...

# bring the moment's position and the user's history to every candidate row
TABLE = TABLE.merge(likes_df[["user_id", "row", "pos", "user_prior_likes"]],
                    left_on=["user_id", "like_row"], right_on=["user_id", "row"], how="left", validate="m:1").drop(columns="row")
assert TABLE["pos"].notna().all()

# likes of each candidate item that happened strictly before the moment
BIG = len(likes_df) + 1
like_keys = np.sort(likes_df["row"].to_numpy(np.int64) * BIG + likes_df["pos"].to_numpy(np.int64))
item_start = TABLE["item_row"].to_numpy(np.int64) * BIG
TABLE["item_prior_likes"] = (np.searchsorted(like_keys, item_start + TABLE["pos"].to_numpy(np.int64), side="left")
                             - np.searchsorted(like_keys, item_start, side="left"))

# item age at the moment, from the first time anybody rated the item
FIRST_SEEN = CLEAN.groupby("item_id")["ts"].min()
TABLE["item_age_days"] = (TABLE["ts"] - TABLE["item_id"].map(FIRST_SEEN)) / DAY

# genres
genre_lists = MOVIES_DF.set_index("item_id")["genres"].str.split("|")
genre_lists = genre_lists.map(lambda names: [g for g in names if g != "(no genres listed)"])
TABLE["main_genre"] = TABLE["item_id"].map(genre_lists.map(lambda names: names[0] if names else "none"))
TABLE["n_genres"] = TABLE["item_id"].map(genre_lists.map(len))

# named buckets
TABLE["user_group"] = pd.cut(TABLE["user_prior_likes"], [-1, 9, 49, 199, np.inf],
                             labels=["new (<10 likes)", "casual (10-49)", "regular (50-199)", "power (200+)"]).astype(str)
TABLE["item_age_group"] = pd.cut(TABLE["item_age_days"] / 365, [0, 1, 5, 15, np.inf], right=False,
                                 labels=["<1 year", "1-5 years", "5-15 years", "15+ years"]).astype(str)
TABLE[["moment_id", "label", "user_prior_likes", "item_prior_likes", "item_age_days", "main_genre", "user_group", "item_age_group"]].head(4)

,moment_id,label,user_prior_likes,item_prior_likes,item_age_days,main_genre,user_group,item_age_group
0,0,1,3,14,6167.506597,Action,new (<10 likes),15+ years
1,0,0,3,0,1498.021042,Drama,new (<10 likes),1-5 years
2,0,0,3,10,5941.236262,Action,new (<10 likes),15+ years
3,0,0,3,1,3548.838009,Western,new (<10 likes),5-15 years


### Self-checks

These are the tests of the pipeline. Each one is an `assert`: if the data were wrong, this cell would stop with a message instead of letting a bad table go to Week 7.

1. **Pandas and Week 5 agree.** `activity`, `age` and `freshness` from the Week 5 code must equal what we compute here from `user_prior_likes` and `item_age_days`. `lifetime` must be `log(1 + item_prior_likes)` divided by a number that is the same for the whole moment.
2. **No peeking.** Every candidate existed before the moment it appears in. The 99 random items were never liked by that user. The `val` period ends before the `test` period starts.
3. **Shape.** Each moment has one liked item first, and 99 others.
4. **A slow, plain-Python check** of a few `item_prior_likes` values.

In [22]:
# 1. two independent computations agree
assert np.allclose(TABLE["activity"], np.minimum(1.0, np.log1p(TABLE["user_prior_likes"]) / np.log1p(ACTIVITY_CAP)))
assert np.allclose(TABLE["age"], np.minimum(1.0, TABLE["item_age_days"] / AGE_CAP_DAYS))
assert np.allclose(TABLE["freshness"], 0.5 ** (TABLE["item_age_days"] / HALF_LIFE_DAYS))
seen = TABLE[TABLE["lifetime"] > 0]
denominator = np.log1p(seen["item_prior_likes"]) / seen["lifetime"]            # must be the same number for every candidate of a moment
spread = denominator.groupby(seen["moment_id"]).transform("max") - denominator.groupby(seen["moment_id"]).transform("min")
assert (spread < 1e-6).all()

# 2. no peeking
assert (TABLE["item_age_days"] > 0).all(), "a candidate that did not exist yet at the moment of the like"
liked_pairs = set(zip(likes_df["user_id"], likes_df["row"]))
negatives = TABLE[TABLE["label"] == 0]
assert not any(pair in liked_pairs for pair in zip(negatives["user_id"], negatives["item_row"])), "a negative the user liked at some point"
assert TABLE.loc[TABLE["split"] == "val", "ts"].max() < T_TEST <= TABLE.loc[TABLE["split"] == "test", "ts"].min()

# 3. shape
labels = TABLE["label"].to_numpy().reshape(-1, CANDIDATES + 1)
assert (labels[:, 0] == 1).all() and (labels[:, 1:] == 0).all(), "each moment: the liked item first, then 99 others"

# 4. a slow check in plain Python
rng = np.random.default_rng(0)
for index in rng.choice(len(TABLE), 5, replace=False):
    r = TABLE.iloc[index]
    slow = sum(1 for k, like in enumerate(LIKES) if like.row == r["item_row"] and k < r["pos"])
    assert slow == r["item_prior_likes"], (index, slow, r["item_prior_likes"])
print("All self-checks passed.")

All self-checks passed.


### Was the leaky column really that tempting?

We kept a copy of `item_total_likes` just for this. **HitRate@10** asks: in how many exam questions does the liked item land in the top 10 of the 100 candidates? Guessing gives about 0.10.

The fair comparison is between **the same count computed two ways**: `item_prior_likes` (likes of the item *before* the moment, which the live app can know) and `item_total_likes` (likes of the item in the *whole log*). The second one also counts the very like we are trying to predict, and every like that comes after it. We also show `popularity`, the Week 5 signal (likes in the last year), for reference.

In [23]:
def hit_rate_at_10(score):
    """Share of exam questions in which the liked item is in the top 10 of 100. Ties count against us."""
    s = np.asarray(score, dtype=float).reshape(-1, CANDIDATES + 1)
    others_at_least_as_good = (s[:, 1:] >= s[:, :1]).sum(axis=1)
    return float((others_at_least_as_good < K).mean())


leaky_score = TABLE["item_id"].map(LEAKY_TOTAL_LIKES)
random_score = np.random.default_rng(SEED).random(len(TABLE))
print(f"random scores                       : HitRate@10 = {hit_rate_at_10(random_score):.3f}")
print(f"popularity (last year, past only)   : HitRate@10 = {hit_rate_at_10(TABLE['popularity']):.3f}")
print(f"item_prior_likes (count, past only) : HitRate@10 = {hit_rate_at_10(TABLE['item_prior_likes']):.3f}")
print(f"item_total_likes (same count, whole log) : HitRate@10 = {hit_rate_at_10(leaky_score):.3f}   <- better only because it contains the answer")

random scores                       : HitRate@10 = 0.097
popularity (last year, past only)   : HitRate@10 = 0.580
item_prior_likes (count, past only) : HitRate@10 = 0.609
item_total_likes (same count, whole log) : HitRate@10 = 0.685   <- better only because it contains the answer


`item_total_likes` and `item_prior_likes` are the same count. The only difference is that the first one includes the like we are asked to predict and the likes that come later, and that alone is worth several points of HitRate. It makes any model that reads it look better than it is, **for a reason that disappears the day the app goes live**. Whenever a result looks too good, ask first: *which column knows something that, at prediction time, nobody could know?*

## Step 9 — Save the feature table and fingerprint it

The final table keeps what Week 7 needs: identifiers, the six signals the model reads, the label, and the slicing columns. We round the signals to 6 decimals (so tiny floating-point differences between machines cannot change the file) and save it as **Parquet**, a compact binary table format that remembers the column types.

Then we compute a **hash**, a short fingerprint of the table's contents. Same data in, same fingerprint out. If your fingerprint matches the one printed below, your table is byte-for-byte the table everybody else has, which is what *reproducible* means.

In [24]:
COLUMNS = ["moment_id", "split", "user_id", "item_id", "ts", "label"] + FEATURES + \
          ["user_prior_likes", "item_prior_likes", "item_age_days", "user_group", "item_age_group", "main_genre", "n_genres"]
FEATURE_TABLE = TABLE[COLUMNS].copy()
FEATURE_TABLE[FEATURES + ["item_age_days"]] = FEATURE_TABLE[FEATURES + ["item_age_days"]].round(6)
FEATURE_TABLE = FEATURE_TABLE.sort_values(["moment_id", "label"], ascending=[True, False]).reset_index(drop=True)

FEATURE_FILE = "feature_table.parquet"
FEATURE_TABLE.to_parquet(FEATURE_FILE, index=False)

fingerprint = hashlib.sha256(FEATURE_TABLE.to_csv(index=False, float_format="%.6f", lineterminator="\n").encode("utf-8")).hexdigest()
EXPECTED_HASH = "977334c6261823d4847242cd1220e21595411fb5934b2d50931d60dcdc1de87b"

print(f"saved {FEATURE_FILE}: {len(FEATURE_TABLE):,} rows x {FEATURE_TABLE.shape[1]} columns, {Path(FEATURE_FILE).stat().st_size / 1e6:.1f} MB")
print("fingerprint:", fingerprint)
print("expected   :", EXPECTED_HASH)
print("MATCH: your table is identical to the reference table." if fingerprint == EXPECTED_HASH else
      "DIFFERENT from the reference. Do the self-checks above all pass? If yes, your library versions probably differ slightly: it is still usable, but tell your instructor.")
FEATURE_TABLE.head()

saved feature_table.parquet: 356,100 rows x 19 columns, 11.3 MB
fingerprint: 977334c6261823d4847242cd1220e21595411fb5934b2d50931d60dcdc1de87b
expected   : 977334c6261823d4847242cd1220e21595411fb5934b2d50931d60dcdc1de87b
MATCH: your table is identical to the reference table.


,moment_id,split,user_id,item_id,ts,label,freshness,affinity,popularity,activity,lifetime,age,user_prior_likes,item_prior_likes,item_age_days,user_group,item_age_group,main_genre,n_genres
0,0,val,376,376,1364932581,1,0.020159,0.15,0.000000,0.222999,0.518748,0.675891,3,14,6167.506597,new (<10 likes),15+ years,Action,2
1,0,val,376,65037,1364932581,0,0.387413,0.10,0.000000,0.222999,0.000000,0.164167,3,0,1498.021042,new (<10 likes),1-5 years,Drama,1
2,0,val,376,1254,1364932581,0,0.023264,0.10,0.000000,0.222999,0.459336,0.651094,3,10,5941.236262,new (<10 likes),15+ years,Action,4
3,0,val,376,5699,1364932581,0,0.105774,0.00,0.000000,0.222999,0.132778,0.388914,3,1,3548.838009,new (<10 likes),5-15 years,Western,1
4,0,val,376,72407,1364932581,0,0.466669,0.04,0.076923,0.222999,0.132778,0.131943,3,1,1203.982072,new (<10 likes),1-5 years,Drama,5


## The audit trail, in one table

In [25]:
pd.DataFrame(AUDIT)

,step,rows before,rows after,removed
0,1. fix types (nothing removed),107388,107388,0
1,2. drop incomplete rows,107388,105876,1512
2,3. drop impossible timestamps,105876,104868,1008
3,4. drop items not in movies.csv,104868,104364,504
4,5a. drop exact repeats,104364,102348,2016
5,"5b. keep the first event per (user, item)",102348,100836,1512


## What to do now

1. **Download `feature_table.parquet` and keep it.** In Colab, the next cell starts the download (or use the file browser on the left). Colab deletes its files when the runtime is recycled, so a copy on your computer or in your Drive is the safe place.
2. **Bring it to Week 7.** The Week 7 notebook asks for this file at the start. If you lost it, no problem: run this notebook again. The pipeline is deterministic, so you get the same file and the same fingerprint.
3. **Nothing else.** No submission, no grade, and no changes to InstaKill this week.

Questions worth thinking about before the next lecture (no answers to hand in):

- Which damage in Step 0 would have been the hardest to notice if the table had been 100 times bigger and you could not look at it?
- If the retries had changed the rating, which one is "right"? How would you decide?
- Why does it matter that the cleaned table matches the original *row for row*, and not just "about the same on average"?

In [26]:
try:
    from google.colab import files
    files.download(FEATURE_FILE)
except ImportError:
    print(f"Running locally: {FEATURE_FILE} is in {Path('.').resolve()}")

Running locally: feature_table.parquet is in C:\GithubProjects\ProgrammingForAi\ProgrammingForAI\lab-06
